In [1]:
# %%capture
import pandas as pd

In [2]:
import gcsfs
filesystem = gcsfs.GCSFileSystem()

/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.cloud._storage_v2 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud._storage_v2 past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/google/ap

In [3]:
import process_ntd
import agency_report_chart
import altair as alt
alt.data_transformers.disable_max_rows()

/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


DataTransformerRegistry.enable('default')

In [4]:
df_capex, df_opex, df_funding, df_agency_mode_efficiency_opex, df_agency_mode_efficiency_capex = (
    process_ntd.load_processed_data_for_agency_report())

In [5]:
# %%capture parameters
# rtpa

In [6]:
rtpa_name = "Metropolitan Transportation Commission"

In [7]:
profile, sel = agency_report_chart.rtpa_profile_chart(df_opex, rtpa_name)
profile

alt.VConcatChart(...)

### 1.1 Agency Trends in Spending, Funding, and Service

In [8]:
datasets = {
    "unlinked_passenger_trips": (df_opex, "Unlinked Passenger Trips"),
    "operating_expenses_total": (df_opex, "Total Operating Expenses"),
    "total_capital_expenditures": (df_capex, "Total Capital Expenditures"),
    "operating_total": (df_funding, "Total Operating Funding"),
    "capital_total": (df_funding, "Total Capital Funding"),
    "vehicle_revenue_hours": (df_opex, "Vehicle Revenue Hours"),
    "vehicle_revenue_miles": (df_opex, "Vehicle Revenue Miles")
}

dashboard, agency_sel = agency_report_chart.rtpa_ntd_value_profile(datasets, rtpa_name)
dashboard

alt.VConcatChart(...)

### 1.2 Expenditure Composition by Mode for Selected Agency and Year

In [9]:
# Operating expenditure
opex_chart, opex_agency_sel, opex_year_sel = agency_report_chart.rtpa_breakdown_chart(
    df_opex, rtpa_name,
    component_columns=[
        "operating_expenses_vehicle_operations",
        "operating_expenses_vehicle_maintenance",
        "operating_expenses_nonvehicle_maintenance",
        "operating_expenses_general_administration",
    ],
    group_column="mode_group", normalize=True,
    title="Operating Expenditure Composition by Mode Group",
)

# Capital expenditure
capex_chart, capex_agency_sel, capex_year_sel = agency_report_chart.rtpa_breakdown_chart(
    df_capex, rtpa_name,
    component_columns=[
        "rolling_stock_expenditures",
        "facilities_expenditures",
        "other_expenditures",
    ],
    group_column="mode_group", normalize=True,
    title="Capital Expenditure Composition by Mode Group",
)

# Combine and style
charts = [opex_chart, capex_chart]

breakdown_by_mode = (
    alt.hconcat(*charts, spacing=25)
    .configure_view(
        strokeWidth=0,
        fill=agency_report_chart.NEUTRAL_COLORS["background"],
    )
    .configure_axis(**agency_report_chart.ALT_AXIS_CONFIG)
    .configure_legend(**agency_report_chart.ALT_LEGEND_CONFIG)
    .configure_title(font=agency_report_chart.CHART_FONT)
)

breakdown_by_mode

/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)


alt.HConcatChart(...)

### 1.3 Funding Composition for Selected Agency and Year

In [10]:
funding_components = {
    "Operating Funding": ["operating_federal", "operating_state", "operating_local", "operating_other"],
    "Capital Funding": ["capital_federal", "capital_state", "capital_local", "capital_other"],
}

funding_chart, funding_agency_sel, funding_year_sel = agency_report_chart.rtpa_funding_composition(
    df_funding, rtpa_name, components=funding_components,
    title="Operating and Capital Funding Composition",
)

funding_chart = (
    funding_chart
    .configure_view(strokeWidth=0, fill=agency_report_chart.NEUTRAL_COLORS["background"])
    .configure_axis(**agency_report_chart.ALT_AXIS_CONFIG)
    .configure_legend(**agency_report_chart.ALT_LEGEND_CONFIG)
    .configure_title(font=agency_report_chart.CHART_FONT)
)

funding_chart

alt.Chart(...)

### 1.4 Operating Cost Efficiency by Mode for Selected Year and Reporting Module 

In [11]:
rank_chart = agency_report_chart.create_grouped_ranking_chart(
    df=df_agency_mode_efficiency_opex,
    metrics={
        "opex_per_vrh": "Opex Cost per VRH ($)",
        "opex_per_vrm": "Opex Cost per VRM ($)",
        "opex_per_upt": "Opex per Trip ($)"
    },
    groups=["Urban", "Rural"],
    group_col="peer_group",
    filter_col="rtpa_name",
    filter_value=rtpa_name
)
rank_chart

alt.VConcatChart(...)

### 1.5 OPEX Mode Performance (HHI, Spending Per Capita and Volatility)

In [12]:
agency_opex_dashboard = agency_report_chart.plot_agency_opex_dashboard(
    df_opex,
    rtpa=rtpa_name,
    type_col="reporting_module",
    dashboard_title="Agency Opex Performance"
)
agency_opex_dashboard

alt.VConcatChart(...)

### 1.6 CAPEX Mode Performance (HHI, Spending Per Capita and Volatility)

In [13]:
agency_capex_dashboard = agency_report_chart.plot_agency_opex_dashboard(
    df_capex,
    rtpa=rtpa_name,
    type_col="reporting_module",
    dashboard_title="Agency Capex Performance"
)
agency_capex_dashboard

alt.VConcatChart(...)

### 1.7 Agency Operating and Service Performance Growth

In [14]:
growth_chart = agency_report_chart.agency_opex_service_growth(
    dataframes={
        "opex": df_opex,
        "capex": df_capex,
    },
    rtpa=rtpa_name,
    metrics={
        "operating_expenses_total_real": ("opex", "Real OPEX"),
        "total_capital_expenditures_real": ("capex", "Real CAPEX"),
        "vehicles_operated_in_maxiumum_service": ("opex", "VOMS"),
        "vehicle_revenue_hours": ("opex", "Revenue Hours"),
        "vehicle_revenue_miles": ("opex", "Revenue Miles"),
        "unlinked_passenger_trips": ("opex", "UPT"),
        "fare_revenue_real": ("opex", "Real Fare Revenue"),
    },
    width=650,
    height=320
)
growth_chart

alt.VConcatChart(...)

### 1.8 Farebox Recovery and Subsidy Relationship

In [8]:
farebox_subsidy_chart = agency_report_chart.create_farebox_subsidy_scatter(
    df=df_agency_mode_efficiency_opex,
    rtpa_name=rtpa_name,
    width=750,
    height=500,
)
farebox_subsidy_chart

alt.VConcatChart(...)

In [16]:
chart, year_sel = agency_report_chart.rtpa_vs_state_funding_share(
    expense_sources=[
        (df_opex, ["operating_expenses_total"]),
        (df_capex, ["total_capital_expenditures"]),
    ],
    df=df_funding,
    components={
        "Federal": ["operating_federal", "capital_federal"],
        "State": ["operating_state", "capital_state"],
        "Local": ["operating_local", "capital_local"],
        "Other": ["operating_other", "capital_other"],
    },
    rtpa=rtpa_name,
    rtpa_column="rtpa_name",
    id_column="ntd_id",
    year_column="year",
    agency_column="source_agency",
)

chart

alt.VConcatChart(...)